# W2 · From Conditional Independence to Message Passing

**Question:** When do local computations recover a global posterior?

Open in a CPU runtime, save a copy in Drive, and run cells from top to bottom. Supporting code is downloaded automatically from a fixed course revision and checked by SHA-256. No dataset upload or GPU is required. All models are explicit synthetic teaching examples. Prediction/fill/debug cells are independent of the runnable demonstrations; skipping an unfinished exercise does not break later cells.

[Lecture](https://github.com/lunalab-ai/advML/blob/main/course/notion/w2/w2-graphical-models.md) · [Project and 10-point rubric](https://github.com/lunalab-ai/advML/blob/main/course/notion/w2/w2-project.md)

The official pyprobml examples informed the small-joint verification approach. This lab is an independent, small NumPy implementation; it does not install the full upstream research stack.


## Shared API: read before calling

[Complete API reference](https://github.com/lunalab-ai/advML/blob/2026-fall-w2-explained/src/API.md)

### [PairwiseModel · definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w2-explained/src/advml_pgm.py#L43)

Finite undirected model represented by log potentials, not fitted parameters.
log_unary: numeric (N,K) array, N nodes and K states per node, N,K >= 1.
log_edges: dict {(u,v): (K,K) numeric array}, indexed [state_u,state_v].
Node IDs are 0..N-1 and states are 0..K-1. Reverse keys are transposed
to canonical order; duplicate undirected edges and self edges are invalid.
The unnormalized log joint sums one unary entry per node and one edge entry
per edge. Factors need not be normalized probability tables. Negative infinity
encodes a hard zero; NaN and positive infinity raise ValueError. Arrays are
copied at construction. Methods perform inference/conditioning; no training
data or fit method is involved. Example: PairwiseModel(np.zeros((2,2)),
{(0,1):np.log([[2.,1.],[1.,2.]])}).

### [PairwiseModel.edge · definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w2-explained/src/advml_pgm.py#L98)

Return the log edge table oriented from node u to node v.
u,v: distinct node IDs of an existing edge. Output shape (K,K), with u
states in rows and v states in columns. Missing edges raise KeyError.
The returned array may share storage with the model: treat it as read-only.
edge(1,0) is edge(0,1).T, not a different undirected factor.

### [PairwiseModel.condition · definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w2-explained/src/advml_pgm.py#L106)

Return a new model clamped to evidence without changing this model.
evidence: dict {node_id: observed_state}, IDs in 0..N-1, states in 0..K-1.
Invalid indices raise ValueError. Inconsistent total mass is detected by
the inference routine. Nonselected unary states receive -inf; the selected
state keeps its original scale. The graph and N remain unchanged: clamping
a cycle does not structurally turn it into a tree. Evidence probability is
exp(log_z_clamped - log_z_original), not exp(log_z_clamped) in general.

### [enumerate_exact · definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w2-explained/src/advml_pgm.py#L124)

Enumerate the complete joint as an independent small-model oracle.
model: PairwiseModel. max_states: maximum K**N assignments, default 262144.
Returns dict: marginals (N,K) normalized probabilities; log_z float;
states (K**N,N) integer assignments; probabilities (K**N,) normalized
joint weights in corresponding order. No input is mutated. Excess states
or zero total mass raise ValueError. Runtime O(K**N*(N+E)); joint storage
is exponential. Use for verification, not large graphs. For clamped models,
log_z retains unnormalized evidence mass and is not generally log $P(e)$.

### [tree_sum_product · definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w2-explained/src/advml_pgm.py#L149)

Run exact two-pass log-domain sum-product on a connected tree.
model: PairwiseModel (single node allowed). root: starting node, default 0.
Returns dict marginals (N,K), log_z float, and log_messages dict whose
(u,v) entry has K values indexed by the receiver v's state. Messages
retain scale so log_z is recoverable. Model is not changed. Invalid root,
a cycle/disconnected graph, or zero total mass raises ValueError.
Time O(N*K**2) at bounded degree, with O(K*sum(degree**2)) extra work
from explicitly recomputing neighboring products. Different roots should
produce the same marginals up to floating-point tolerance.

### [loopy_sum_product · definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w2-explained/src/advml_pgm.py#L202)

Run synchronous approximate sum-product with probability-space damping.
model: PairwiseModel. damping: OLD-message weight in [0,1), default 0.
max_iter: positive iteration limit, default 300. tol: positive stopping
threshold on maximum elementwise normalized-message change, default 1e-10.
Returns dict marginals (N,K), residuals (iterations,), converged bool,
and iterations int. Inputs are unchanged; messages start uniformly.
Each update uses the previous iteration, not partially updated neighbors.
No log_z estimate is claimed. A small residual is not a bound on marginal
error on cycles. Invalid settings/impossible local mass raise ValueError.
Compare accuracy with enumerate_exact only on tractable models.

### [ising_model · definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w2-explained/src/advml_pgm.py#L246)

Construct a binary PairwiseModel from fields and couplings.
fields: length-N numeric array of local fields h_i. couplings: dict
{(u,v): J_uv} of scalar edge strengths. State 0 maps to spin -1 and
state 1 to +1. Unary log potential is h_i*s_i; edge log potential is
J_uv*s_u*s_v, so positive J favors agreement. Returns a new model;
no fitting or random sampling occurs. Example: ising_model(np.zeros(2),
{(0,1):0.8}) favors equal states without a local preference.

### [max_marginal_tv · definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w2-explained/src/advml_pgm.py#L259)

Return max_i 0.5*sum_k(abs(p[i,k]-q[i,k])) as a float.
p,q: matching normalized probability arrays of shape (N,K). The caller
is responsible for nonnegative, unit-sum rows. Shape mismatch raises
ValueError. Inputs are not changed. This compares single-node marginals,
not joint distributions and not consecutive messages. Example: rows
[0.2,0.8] versus [0.5,0.5] have TV 0.3.


In [ ]:
from pathlib import Path
import hashlib, importlib.util, sys, tempfile, urllib.request, subprocess
for package, version in [("numpy", "2.2.6"), ("matplotlib", "3.10.3")]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}=={version}"])
CODE_REF = "2026-fall-w2-explained"
EXPECTED_SHA256 = "97c68026a1024694ec97a632449aa6ada97d18fe0a8fb94212d260818fbf0a28"
CODE_URL = f"https://raw.githubusercontent.com/lunalab-ai/advML/{CODE_REF}/src/advml_pgm.py"
local = Path("src/advml_pgm.py")
if local.is_file() and hashlib.sha256(local.read_bytes()).hexdigest() == EXPECTED_SHA256:
    module_path = local.resolve()
    print("Using verified course-checkout code.")
else:
    module_path = Path(tempfile.gettempdir()) / f"advml_pgm_{EXPECTED_SHA256[:16]}.py"
    if not module_path.is_file() or hashlib.sha256(module_path.read_bytes()).hexdigest() != EXPECTED_SHA256:
        for attempt in range(3):
            try:
                with urllib.request.urlopen(CODE_URL, timeout=30) as response:
                    payload = response.read()
                if hashlib.sha256(payload).hexdigest() != EXPECTED_SHA256:
                    raise RuntimeError("Course module checksum mismatch. Stop and report this; do not bypass the check.")
                module_path.write_bytes(payload)
                break
            except Exception as exc:
                if attempt == 2:
                    raise RuntimeError("Could not prepare the fixed course module. Check the connection and rerun this cell; the complete public course checkout is the offline alternative.") from exc
    print("Using verified fixed-revision public code (download/cache).")
spec = importlib.util.spec_from_file_location("advml_pgm", module_path)
pgm = importlib.util.module_from_spec(spec)
sys.modules["advml_pgm"] = pgm
spec.loader.exec_module(pgm)
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from advml_pgm import PairwiseModel, enumerate_exact, tree_sum_product, loopy_sum_product, ising_model, max_marginal_tv
print("Python", sys.version.split()[0], "NumPy", np.__version__, "Matplotlib", matplotlib.__version__)
print("Reference:", CODE_REF, "SHA-256:", EXPECTED_SHA256)
SEED = 20260908
rng = np.random.default_rng(SEED)


## 1. Selection can create dependence

Before running: A and B are independent Bernoulli variables with probabilities 0.2 and 0.3. We retain a case exactly when $S=(A\lor B)$. Predict $P(A=1)$, $P(A=1 \mid S=1)$, and $P(A=1 \mid S=1, B=1)$. Explain which path is activated by observing S.


In [ ]:
joint = np.zeros((2, 2, 2))  # axes: A, B, S
for aa in range(2):
    for bb in range(2):
        joint[aa, bb, int(aa or bb)] = [0.8, 0.2][aa] * [0.7, 0.3][bb]
prior_ab = joint.sum(axis=2)
selected_ab = joint[:, :, 1] / joint[:, :, 1].sum()
print("Prior joint:\n", prior_ab)
print("Selected joint:\n", selected_ab)
print("P(A=1):", prior_ab[1].sum())
print("P(A=1 | S=1):", selected_ab[1].sum())
print("P(A=1 | S=1, B=1):", selected_ab[1, 1] / selected_ab[:, 1].sum())
np.testing.assert_allclose(prior_ab, np.outer(prior_ab.sum(1), prior_ab.sum(0)))
assert not np.allclose(selected_ab, np.outer(selected_ab.sum(1), selected_ab.sum(0)))


### Predict a descendant observation

Now observe a noisy descendant $D$ of $S$ instead of $S$ itself. Let $P(D=1 \mid S=0)=0.05$ and $P(D=1 \mid S=1)=0.9$. Is the collider path active? Does the size of dependence have to equal that obtained from observing $S$ exactly?


In [ ]:
weighted_ab = (joint * np.array([0.05, 0.9])[None, None, :]).sum(axis=2)
given_d = weighted_ab / weighted_ab.sum()
independence_gap = np.max(np.abs(given_d - np.outer(given_d.sum(1), given_d.sum(0))))
print("Joint given D=1:\n", given_d, "\nIndependence gap:", independence_gap)
assert independence_gap > 1e-3


## 2. Eight states are enough to audit the worked example

The prior is [0.6,0.4]. Transition tables have source states in rows and destination states in columns. The likelihood at C is [0.3,0.7]. Predict the posterior at B before running. The local evidence need not sum to one across states; it is a likelihood for a fixed observation.


In [ ]:
T_ab = np.array([[0.9, 0.1], [0.2, 0.8]])
T_bc = np.array([[0.8, 0.2], [0.1, 0.9]])
evidence_c = np.array([0.3, 0.7])
chain = PairwiseModel(np.log([[0.6, 0.4], [1.0, 1.0], evidence_c]),
                      {(0, 1): np.log(T_ab), (1, 2): np.log(T_bc)})
oracle = enumerate_exact(chain)
bp = tree_sum_product(chain)
print("Exact marginals [A,B,C]:\n", oracle["marginals"])
print("BP marginals:\n", bp["marginals"])
print("Evidence probability:", np.exp(bp["log_z"]))
np.testing.assert_allclose(bp["marginals"], oracle["marginals"], atol=1e-12)
np.testing.assert_allclose(bp["marginals"][1], [0.484375, 0.515625])
assert np.isclose(np.exp(bp["log_z"]), 0.4864)


### Check conditional independence with the full joint

Condition on B=1. Compare the conditional joint of A and C with the product of its marginals. This is a test of this distribution; the graph-based claim for the whole model family needs the derivation in the lecture.


In [ ]:
joint_abc = oracle["probabilities"].reshape(2, 2, 2)
ac_given_b = joint_abc[:, 1, :] / joint_abc[:, 1, :].sum()
np.testing.assert_allclose(ac_given_b, np.outer(ac_given_b.sum(1), ac_given_b.sum(0)), atol=1e-12)
print("Conditional independence check passed.")


## 3. Fill the message contraction

**Goal:** identify which state gets summed out. Compute the message C -> B using `T_bc` and `evidence_c`. Set `student_message` to your two-entry array. Hint: the result has one entry per B state; write the scalar expression for B=0 first. Self-check: the validator compares with an independent explicit sum. Leaving `None` lets the rest of the lab run.


In [ ]:
student_message = None  # replace with your expression
if student_message is None:
    print("Exercise left open. Predict the shape and complete it before checking.")
else:
    manual_check = np.array([sum(T_bc[b, c] * evidence_c[c] for c in range(2)) for b in range(2)])
    np.testing.assert_allclose(student_message, manual_check)
    print("Message contraction check passed.")


## 4. Diagnose a plausible transpose bug

An implementation uses the expression below. It returns the right shape, but is it the right message? Set `student_fix` to your corrected result and explain the indexing error in words. Hint: compare the meaning of rows and columns; matching shape alone is not evidence of correctness.


In [ ]:
buggy_message = T_bc.T @ evidence_c
print("Buggy value:", buggy_message)
student_fix = None
if student_fix is not None:
    np.testing.assert_allclose(student_fix, np.exp(bp["log_messages"][(2, 1)]))
    assert not np.allclose(student_fix, buggy_message)
    print("Fix validated. Write why the incorrect transpose changes the model.")
else:
    print("Debugging exercise open; later experiments use the validated library.")


## 5. Filtering versus smoothing

These observation symbols are a synthetic sensor sequence. States persist but observations are noisy. Predict whether observations near the end can change a smoothed belief near the beginning. For an online deployment, which output is available without future leakage?


In [ ]:
observations = np.array([0, 0, 1, 0, 1, 1, 1, 0])
transition = np.array([[0.92, 0.08], [0.12, 0.88]])
emission = np.array([[0.85, 0.15], [0.2, 0.8]])  # [state, symbol]
likelihoods = emission[:, observations].T
unary = np.log(likelihoods)
unary[0] += np.log([0.5, 0.5])
hmm = PairwiseModel(unary, {(t, t+1): np.log(transition) for t in range(len(observations)-1)})
smoothed = tree_sum_product(hmm)["marginals"]
filtered = []
alpha = np.array([0.5, 0.5])
for t, likelihood in enumerate(likelihoods):
    predicted = alpha if t == 0 else alpha @ transition
    alpha = likelihood * predicted
    alpha /= alpha.sum()
    filtered.append(alpha.copy())
filtered = np.array(filtered)
np.testing.assert_allclose(smoothed, enumerate_exact(hmm)["marginals"], atol=1e-12)
np.testing.assert_allclose(filtered[-1], smoothed[-1], atol=1e-12)
plt.figure(figsize=(9,3))
plt.plot(filtered[:,1], "o-", label="Filtered P(z_t=1 | y_1:t)")
plt.plot(smoothed[:,1], "s-", label="Smoothed P(z_t=1 | all y)")
plt.xlabel("Time index"); plt.ylabel("State-1 probability"); plt.ylim(0,1); plt.legend(); plt.tight_layout(); plt.show()


## 6. A converged answer can still be inaccurate

Close a chain into a triangle. Positive J favors agreement; a field at one node breaks the zero-field symmetry. Predict whether the final residual bounds the posterior error. The two quantities below measure different things; do not compare them as interchangeable tolerances.


In [ ]:
cycle = ising_model(np.array([0.4, 0.0, 0.0]), {(0,1):1.3, (1,2):1.3, (0,2):1.3})
truth = enumerate_exact(cycle)
approx = loopy_sum_product(cycle, damping=0.2)
error = max_marginal_tv(truth["marginals"], approx["marginals"])
print("Converged:", approx["converged"], "Iterations:", approx["iterations"])
print("Final message residual:", approx["residuals"][-1])
print("Maximum one-node TV error:", error)
print("Exact:\n", truth["marginals"], "\nApproximate:\n", approx["marginals"])
assert approx["converged"] and error > 0.05
plt.figure(figsize=(8,3)); plt.semilogy(approx["residuals"], label="Message residual")
plt.axhline(error, color="tab:red", linestyle="--", label="Final marginal TV error (different metric)")
plt.xlabel("Iteration"); plt.legend(); plt.tight_layout(); plt.show()


### Controlled variation

Change coupling strength while keeping fields and graph fixed. Does damping improve accuracy, stability, both or neither? A comparison must keep the model fixed when comparing algorithms. Print nonconverged cases rather than silently removing them.


In [ ]:
records = []
for coupling in [0.2, 0.8, 1.3]:
    model = ising_model(np.array([0.4, 0.0, 0.0]), {(0,1):coupling, (1,2):coupling, (0,2):coupling})
    exact = enumerate_exact(model)
    for damping in [0.0, 0.3, 0.6]:
        fit = loopy_sum_product(model, damping=damping, max_iter=1000, tol=1e-9)
        row = dict(coupling=coupling, damping=damping, converged=bool(fit["converged"]),
                   iterations=fit["iterations"], residual=float(fit["residuals"][-1]),
                   marginal_tv=max_marginal_tv(exact["marginals"], fit["marginals"]))
        records.append(row)
        print(row)


## 7. Numerical and structural checks

Multiply a factor by a large constant. Predict what should happen to normalized marginals and log Z. Then try impossible evidence in an equality-constrained model. A uniform fallback would conceal an invalid conditional distribution.


In [ ]:
shifted = PairwiseModel(chain.log_unary + 1000, chain.log_edges)
shifted_fit = tree_sum_product(shifted)
np.testing.assert_allclose(shifted_fit["marginals"], bp["marginals"], atol=1e-10)
assert np.isclose(shifted_fit["log_z"] - bp["log_z"], 3000)
equality = PairwiseModel(np.zeros((2,2)), {(0,1):np.array([[0,-np.inf],[-np.inf,0]])})
try:
    tree_sum_product(equality.condition({0:0, 1:1}))
except ValueError as exc:
    print("Expected impossible-evidence error:", exc)
else:
    raise AssertionError("An impossible event must not return a posterior.")
print("Scale and impossible-evidence checks passed.")


## Write your conclusion before opening the quiz

1. Give one conditional-independence claim and its graph-based justification.
2. Explain why C -> B is a function of B even though C sends it.
3. State one exactness guarantee and the assumptions that make it valid.
4. Describe the evidence that refutes “a tiny message residual proves accurate marginals.”
5. Propose a new experiment that would make the counterexample more informative.

The assessed project goes beyond these supplied examples: you must derive and implement the cycle correction, create your own experiment families, and evaluate a justified extension. [Project specification](https://github.com/lunalab-ai/advML/blob/main/course/notion/w2/w2-project.md).

References: Murphy, *Advanced Topics*, Chapters 4, 7 and 9; [official pyprobml](https://github.com/probml/pyprobml/tree/master/notebooks/book2). The figures produced here are generated from this notebook's synthetic examples.


In [ ]:
np.testing.assert_allclose(bp["marginals"], oracle["marginals"], atol=1e-12)
assert approx["converged"] and error > 0.05
